# 06 — Supervised ranker (walk 1 MVP)

LightGBM `LGBMRanker(lambdarank)` over PCA text + structured + macro + aux text
features. Walk-1 only (train 2002-2007, val 2008, test 2009). Outputs land in
`artifacts/ranker/walk-001/`.

**Spec:** `docs/superpowers/specs/2026-05-16-supervised-ranker-design.md`.

Per the `machine-learning` skill: Optuna study persisted, early stopping on val
NDCG@30, no tuning on test, model + HPs + diagnostics versioned together.


## A. Setup

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import optuna
from lightgbm import early_stopping
import matplotlib.pyplot as plt

from src.utils.io import processed_dir, repo_root
from src.utils.ranker import (
    DEFAULT_RANKER_PARAMS,
    assemble_walk_features,
    build_ranker,
    compute_excess_return_buckets,
    evaluate_ranker,
    load_walk_pca,
    project_text_to_pca,
)

WALK_ID = 1
TRAIN_START, TRAIN_END = '2002-01-01', '2007-12-31'
VAL_START,   VAL_END   = '2008-01-01', '2008-12-31'
TEST_START,  TEST_END  = '2009-01-01', '2009-12-31'

PANEL_DIR = processed_dir() / 'training_panel'
EMBED_DIR = processed_dir() / 'finbert_stockday_embed'
OUT_DIR   = repo_root() / 'artifacts' / 'ranker' / f'walk-{WALK_ID:03d}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_OPTUNA_TRIALS = 15
N_BUCKETS = 32
TOP_K = 30
RANDOM_STATE = 42

print(f'walk {WALK_ID}')
print(f'  train: {TRAIN_START} .. {TRAIN_END}')
print(f'  val:   {VAL_START} .. {VAL_END}')
print(f'  test:  {TEST_START} .. {TEST_END}')
print(f'out_dir: {OUT_DIR.relative_to(repo_root())}')

## B. Load PCA + assemble train/val/test feature matrices

Reads the per-year shards in the three windows, projects text vecs through the
walk-1 PCA (n_pca=79), and assembles (X, y_excess, group_sizes, meta) triples
via `assemble_walk_features`. Memory ceiling: ~470k train rows × ~196 cols ≈ 700 MB.

In [ ]:
pca, n_pca = load_walk_pca(WALK_ID)
print(f'PCA loaded: n_components = {n_pca}')


def _load_panel_years(start: str, end: str) -> pd.DataFrame:
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    frames = []
    for y in range(s.year, e.year + 1):
        for p in sorted((PANEL_DIR / f'year={y}').glob('*.parquet')):
            df = pd.read_parquet(p)
            df['date'] = pd.to_datetime(df['date'])
            df = df[(df['date'] >= s) & (df['date'] <= e)]
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def _load_embed_years(start: str, end: str) -> pd.DataFrame:
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    frames = []
    for y in range(s.year, e.year + 1):
        for p in sorted((EMBED_DIR / f'year={y}').glob('*.parquet')):
            df = pd.read_parquet(p, columns=['permno', 'date', 'vec'])
            df['date'] = pd.to_datetime(df['date'])
            df = df[(df['date'] >= s) & (df['date'] <= e)]
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def _assemble(window_start: str, window_end: str, label: str):
    panel = _load_panel_years(window_start, window_end)
    embed = _load_embed_years(window_start, window_end)
    embed_pca = project_text_to_pca(embed, pca)
    X, y_excess, groups, meta = assemble_walk_features(panel, embed_pca)
    print(f'{label}: panel={len(panel):>7}  embed={len(embed):>7}  '
          f'X={X.shape}  groups={len(groups)} Fridays')
    return X, y_excess, groups, meta


X_train, y_train_excess, groups_train, meta_train = _assemble(TRAIN_START, TRAIN_END, 'train')
X_val,   y_val_excess,   groups_val,   meta_val   = _assemble(VAL_START,   VAL_END,   'val  ')
X_test,  y_test_excess,  groups_test,  meta_test  = _assemble(TEST_START,  TEST_END,  'test ')

# Lambdarank labels (32 buckets on per-date excess return).
buckets_train = compute_excess_return_buckets(meta_train, n_buckets=N_BUCKETS).astype(int).values
buckets_val   = compute_excess_return_buckets(meta_val,   n_buckets=N_BUCKETS).astype(int).values

print(f'feature columns: {X_train.shape[1]}')
print(f'first 8 feature cols: {list(X_train.columns[:8])}')
print(f'NaN-rich cols in train (top 5): '
      f'{X_train.isna().mean().sort_values(ascending=False).head(5).to_dict()}')